In [1]:
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import getpass
import subprocess
import time
import importlib
from shutil import copy2

%load_ext autoreload
%autoreload 2

!nvcc --version

### Path to this cloned GitHub repo:
SCRIPT_DIR = os.path.dirname("/home/machenshu/works/heme_binder_diffusion_ff/")  # edit this to the GitHub repo path. Throws an error by default.
assert os.path.exists(SCRIPT_DIR)
sys.path.append(SCRIPT_DIR+"/scripts/utils")
print(SCRIPT_DIR+"/scripts/utils")
print(os.listdir(SCRIPT_DIR+"/scripts/utils"))
#sys.path.insert(0, SCRIPT_DIR+"/scripts/utils")
import utils

ModuleNotFoundError: No module named 'pandas'

# De novo heme binding protein design pipeline using RFdiffusionAA
#### And other ligand binders too, I guess
Indrek Kalvet, PhD (Institute for Protein Design, University of Washington), ikalvet@uw.edu

As implemented in the publication "<i>Generalized Biomolecular Modeling and Design with RoseTTAFold All-Atom"

Link:

This notebook illustrates the design of heme-binding proteins, starting from minimal information (heme + substrate + CYS motif). It should work with minor modifications also for any other ligand.

The pipeline consists of 7 steps:<br>

    0) The protein backbones are generated with RFdiffusionAA
    1) Sequence is designed with proteinMPNN (without the ligand)
    2) Structures are predicted with AlphaFold2
    3) Ligand binding site is designed with LigandMPNN/FastRelax, or Rosetta FastDesign
    4) Sequences surrounding the ligand pocket are diversified with LigandMPNN
    5) Final designed sequences are predicted with AlphaFold2
    6) Alphafold2-predicted models are relaxed with the ligand and analyzed


## Paths of important Python scripts and programs

#### RFdiffusionAA:
Download RFdiffusionAA from here: https://github.com/baker-laboratory/rf_diffusion_all_atom

#### RFjoint inpainting (proteininpainting)
(Optional) Download RFjoint Inpainting here: https://github.com/RosettaCommons/RFDesign

Inpainting is used to further resample/diversify diffusion outputs, and it may also increase AF2 success rates.
<br>

#### AlphaFold2 and LigandMPNN
Other dependencies (ligandMPNN and AlphaFold2) are downloaded as submodules with this GitHub repository.

<br>
<br>
<i>After these evironments are set up and repositories are downloaded please provide paths to the inference scripts in the cell below:</i>

<br>
<br>

#### Python or Apptainer image
Please define in the cell below any Python executables or Apptainer image paths that are able to run the different scripts.<br>

This pipeline is tested to run based on two different Conda environments: `diffusion` and `mlfold`.<br>
Both of these can be set up based on the yml files provided by this repository.<br>
`mlfold` is used for AlphaFold2, and `diffusion` is used for everything else.

<br>

Alternatively, if your system is using Apptainers, you can set them up based on the same yml files, or do it separately. For RFdiffusionAA you can download this Apptainer image:
http://files.ipd.uw.edu/pub/RF-All-Atom/containers/rf_se3_diffusion.sif

If you are setting the environments or containers up without the provided YML files, then the minimum requirements for the different types are:<br>
`af2` Python needs to include jax=0.4.25 and jaxlib=0.4.23<br>
`proteinMPNN` Python needs to include pytorch and prody<br>
`general` Python needs to include pytorch, pyrosetta, prody<br>



In [ ]:
diffusion_script = "/home/machenshu/works/rf_diffusion_all_atom/run_inference.py"  # edit this
inpaint_script = "PATH/TO/RFDesign/inpainting/inpaint.py"  # edit this if needed
proteinMPNN_script = f"{SCRIPT_DIR}/lib/LigandMPNN/run.py"  # from submodule
AF2_script = f"{SCRIPT_DIR}/scripts/af2/af2.py"  # from submodule

#added by Feng, so to check wether they are exist, in case.
if not os.path.exists(diffusion_script):
    print("ERROR: diffusion_script does not exists!!!")
    
assert os.path.exists(proteinMPNN_script)

assert os.path.exists(AF2_script)


### Python and/or Apptainer executables needed for running the jobs
### Please provide paths to executables that are able to run the different tasks.
### They can all be the same if you have an environment with all of the ncessary Python modules in one

# If your added Apptainer does not execute scripts directly,
# try adding 'apptainer run' or 'apptainer run --nv' (for GPU) in front of the command

CONDAPATH = "/home/machenshu/miniconda3"   # edit this depending on where your Conda environments live
PYTHON = {"diffusion": f"{CONDAPATH}/envs/diffusion/bin/python",
          "af2": f"{CONDAPATH}/envs/mlfold/bin/python",
          "proteinMPNN": f"{CONDAPATH}/envs/diffusion/bin/python",
          "ligandMPNN": f"{CONDAPATH}/envs/ligandmpnn_env3/bin/python",
          "general": f"{CONDAPATH}/envs/diffusion/bin/python"}

## Project description and paths

In [ ]:
username = getpass.getuser()  # your username on the running system
EMAIL = f"{username}@bu.edu"  # edit based on your organization. For Slurm job notifications.

PROJECT = "example_Heme_diffusion"

print("Username:", username)
print(f"Email:{EMAIL}")
print(f"Project:{PROJECT}")


### Path where the jobs will be run and outputs dumped
WDIR = "/home/machenshu/works/heme_pipeline/protein_design/project1"

if not os.path.exists(WDIR):
    os.makedirs(WDIR, exist_ok=True)

print(f"Working directory: {WDIR}")

USE_GPU_for_AF2 = True


In [ ]:
# Ligand information
params = [f"{SCRIPT_DIR}/theozyme/HBA/HBA.params"]  # Rosetta params file(s)
print(params)

for p in params:
    if not os.path.exists(p):
        print("Error: missing param file!!")
    else :
        print(f"Checking param file \"{os.path.basename(p)}\"....yes, file found!!!")
        
LIGAND = "HBA"

## 0: Setting up diffusion run

In [ ]:
# Using example PDB file with ligand HBA and protein 7o2g backbone.
## Note: the repository also contains additional HBA conformers with 7o2g and P450 motifs
## in the same directory as a ZIP file.

diffusion_inputs = glob.glob(f"{SCRIPT_DIR}/input/*.pdb")
print(f"Found {len(diffusion_inputs)} PDB files")
print(diffusion_inputs)

In [ ]:
## Setting up general settings for diffusion

DIFFUSION_DIR = f"{WDIR}/0_diffusion"
if not os.path.exists(DIFFUSION_DIR):
    os.makedirs(DIFFUSION_DIR, exist_ok=False)

os.chdir(DIFFUSION_DIR)

N_designs = 50
T_steps = 100

## Edit this config based on motif residues, etc...
config = f"""
defaults:
  - aa

diffuser:
  T: {T_steps}

inference:
  num_designs: {N_designs}
  model_runner: NRBStyleSelfCond
  ligand: '{LIGAND}'

model:
  freeze_track_motif: True

contigmap:
  contigs: ["30-110,A15-15,30-110"]
  inpaint_str: null
  length: "100-140"

potentials:
  guiding_potentials: ["type:ligand_ncontacts,weight:1"] 
  guide_scale: 2
  guide_decay: cubic
"""

estimated_time = 3.5 * T_steps * N_designs  # assuming 3.5 seconds per timestep on A4000 GPU

print(f"Estimated time to produce {N_designs} designs = {estimated_time/60:.0f} minutes")
with open("config.yaml", "w") as file:
    file.write(config)
print(f"Wrote config file to {os.path.realpath('config.yaml')}")

In [ ]:
## Setting up diffusion commands based on the input PDB file(s)
## Diffusion jobs are run in separate directories for each input PDB

commands_diffusion = []
cmds_filename = "commands_diffusion"
diffusion_rundirs = []
print(os.getcwd())

submit_commands={}  # added by Feng to remember the command to submit to run local with subprocess

with open(cmds_filename, "w") as file:
    for p in diffusion_inputs:
        pdbname = os.path.basename(p).replace(".pdb", "")
        os.makedirs(pdbname, exist_ok=True)
        cmd_submit_temp=[f"{PYTHON['diffusion']}", f"{diffusion_script}",f"--config-dir=../",\
              f"--config-name=config.yaml",f"inference.input_pdb={p}",\
              f"inference.output_prefix='./out/{pdbname}_dif'"]#,f" > output.log"] 
        
        cmd = f"cd {pdbname} ; {PYTHON['diffusion']} {diffusion_script} --config-dir=../ "\
              f"--config-name=config.yaml inference.input_pdb={p} "\
              f"inference.output_prefix='./out/{pdbname}_dif' > output.log ; cd ..\n"
        commands_diffusion.append(cmd)
        diffusion_rundirs.append(pdbname)
        file.write(cmd)
        submit_commands.update({pdbname:cmd_submit_temp})

print(f"An example diffusion command that was generated:\n   {cmd}")

print(f"showing the command dictionary:{submit_commands}")


**Feng notes:** Very interestingly, we need to revise the script, "run_inference.py", to move "import numpy" to the beginnng of the module. Otherwise we will see error like:

" b'Error: mkl-service + Intel(R) MKL: MKL_THREADING_LAYER=INTEL is incompatible with libgomp.so.1 library.\n\tTry to import numpy first or set the threading layer accordingly. Set MKL_SERVICE_FORCE_INTEL to force it.\n' "

Google gemini is saying we need to move numpy to the beginning, so 

"In some cases, importing numpy before other libraries that might trigger the MKL service can help resolve the conflict by ensuring MKL is initialized correctly."

But such kind of error won't show if we simply run the script at the command line directly.


Also need to change the file in rf_diffusion_aa folder, "/home/Feng/hg/rf_diffusion_all_atom/config/inference/aa.yaml". We need to add the absolute directory to the weight/parameter file, otherwise the script can not find the model paramter file.


In [ ]:

## Creating a Slurm submit script
## adjust time depending on number of designs and available hardware
submit_script = "submit_diffusion.sh"
utils.create_slurm_submit_script(filename=submit_script, name="diffusion_example", gpu=True, gres="gpu:a4000:1",
                                 mem="8g", N_cores=2, time="1:00:00", email=EMAIL,
                                 array=len(commands_diffusion), array_commandfile=cmds_filename)

print(f"Writing diffusion submission script to {submit_script}")
print(f"{len(commands_diffusion)} diffusion jobs to run\n\n")

print("Start running the command(s).....")

for pdbname, cmd in submit_commands.items():
    if not os.path.exists(DIFFUSION_DIR+"/.done"):
        #set the working directory
        os.chdir(DIFFUSION_DIR+"/"+pdbname)
        print(f"working directory:{os.getcwd()}\n\n")
        print(f"command to run:{cmd}\n\n")
        
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        (output, err) = p.communicate()
        #p.wait()
        #print("**output:", output)
        with open("output.log", "w") as file_log:
            file_log.write(output)
        print("\tshowing error messages:", err)
        os.chdir("..")

        print(f"\n\nGo back to {os.getcwd()}")
    

In [ ]:
print(output)


In [ ]:
## If you're done with diffusion and happy with the outputs then mark it as done
#DIFFUSION_DIR = DIFFUSION_DIR #f"{WDIR}/0_diffusion" <---- don't need to reset it again
os.chdir(DIFFUSION_DIR)

if not os.path.exists(DIFFUSION_DIR+"/.done"):
    with open(f"{DIFFUSION_DIR}/.done", "w") as file:
        file.write(f"Run user: {username}\n")


### Analyzing diffusion outputs
The purpose of this step is to identify diffused backbones that meet certain quality criteria. These scaffolds should be relatively globular (measured by radius of gyration (rog), and longest helix). They should not have clashes between the ligand and the backbone, the ligand should not be too exposed (measured by relative SASA). The termini should not be too close to the ligand (term_mindist), and the backbone should not be too loopy. In the example below we are also looking for backbones that leave some part of the ligand more exposed.

In [ ]:
### Analyzing diffusion outputs for clashes, ligand burial and scaffold quality
## If it's running too slowly consider increasing --nproc

print("Start with the script.............")

analysis_script = f"{SCRIPT_DIR}/scripts/diffusion_analysis/process_diffusion_outputs.py"

print(f"name: {analysis_script}\n\n")

#
print("enumerate the output directories..............\n\n")
print(f" diffusion_rundirs:{diffusion_rundirs} \n")
print(f" DIFFUSION_DIR:{DIFFUSION_DIR} \n")

diffusion_outputs = []
for d in diffusion_rundirs:
    diffusion_outputs += glob.glob(f"{d}/out/*.pdb")

print(f" diffusion output :{diffusion_outputs}")
print("set up the arguents..........\n\n")

# By default I don't use the --analyze flag. As a result the backbones are filtered as the script runs.
# You can set --analyze to True to calculate all scores for all backbones.
# This will slow the analysis down, but you can then filter the backbones separately afterwards.
dif_analysis_cmd_dict = {"--pdb": " ".join(diffusion_outputs),
                        "--ref": f"{SCRIPT_DIR}/input/*.pdb",
                        "--params": " ".join(params),
                        "--term_limit": "15.0",
                        "--SASA_limit": "0.3",  # Highest allowed relative SASA of ligand
                        "--loop_limit": "0.4",  # Fraction of backbone that can be loopy
                        "--ref_catres": "A15",  # Position of CYS in diffusion input
                        "--rethread": True,
                        "--fix": True,
                        "--exclude_clash_atoms": "O1 O2 O3 O4 C5 C10",  # Ligand atoms excluded from clashchecking because they are flexible
                        "--ligand_exposed_atoms": "C45 C46 C47",  # Ligand atoms that need to be more exposed
                        "--exposed_atom_SASA": "10.0",  # minimum absolute SASA for exposed ligand atoms
                        "--longest_helix": "30",
                        "--rog": "30.0",
                        "--partial": None,
                        "--outdir": None,
                        "--traj": "5/30",  # Also random 5 models are taken from the last 30 steps of the diffusion trajectory
                        "--trb": None,
                        "--analyze": False,
                        "--nproc": "1"}


print("\n\n Start building the command string........\n")

analysis_command = f"{PYTHON['general']} {analysis_script}"
for k, val in dif_analysis_cmd_dict.items():
    if val is not None:
        if isinstance(val, list):
            analysis_command += f" {k}"
            analysis_command += " " + " ".join(val)
        elif isinstance(val, bool):
            if val == True:
                analysis_command += f" {k}"
        else:
            analysis_command += f" {k} {val}"
        print(k, val)

#show the analysis command
print(f"\n\nThe analysis command :\n{analysis_command}")

if len(diffusion_outputs) < 550:  #was originally 100
    ## Analyzing locally
    p = subprocess.Popen(analysis_command, shell=True, text=True)
    (output, err) = p.communicate()
else:
    ## Too many structures to analyze.
    ## Running the analysis as a SLURM job.
    submit_script = "submit_diffusion_analysis.sh"
    utils.create_slurm_submit_script(filename=submit_script, name="diffusion_analysis",
                                     mem="8g", N_cores=dif_analysis_cmd_dict["--nproc"], time="0:20:00", email=EMAIL,
                                     command=analysis_command, outfile_name="output_analysis")

diffused_backbones_good = glob.glob(f"{DIFFUSION_DIR}/filtered_structures/*.pdb")

dif_analysis_df = pd.read_csv(f"{DIFFUSION_DIR}/diffusion_analysis.sc", header=0, sep="\\s+")

In [ ]:
## Visualizing the distributions of diffusion analysis metrics
## Plotting design scores
plt.figure(figsize=(12, 12))
for i,k in enumerate(dif_analysis_df.keys()):
    if k in ["description"]:
        continue
    plt.subplot(4, 3, i+1)
    plt.hist(dif_analysis_df[k])
    plt.title(k)
    plt.xlabel(k)
plt.tight_layout()
plt.show()

It is highly advised that you manually inspect the filtered diffusion outputs before continuing with the pipeline.
While the filters attempt to pick out the most offending designs then nothing beats your own intuition and judgement.

If you would like to perform RFjoint Inpainting on the diffusion outputs, please go to the [inpainting section](#inpainting)

## 
## --------------------------------------------------------------------------------------------------------------------
<a id='inpainting'></a>

## RFjoint Inpainting diversification of diffusion outputs
This part of the pipeline branches off right after diffusion.

The purpose of inpainting diversification is to resample the loop regions of diffused backbones. Any region that is close to the ligand will not be touched because inpainting model does not see the ligand. Resampling is done with variable sequence length.

For some reason the AF2 success rates may go significantly up when performing inpainting on diffused backbones.


In [ ]:
WDIR = "/home/machenshu/works/heme_binder_diffusion_ff"
DIFFUSION_DIR = f"{WDIR}/protein_design/project1/0_diffusion"
SCRIPT_DIR = f"{WDIR}"
PYTHON = {"general": "python"}

print("WDIR =", WDIR)
print("DIFFUSION_DIR =", DIFFUSION_DIR)

In [ ]:
import os, glob
fs_dir = f"{DIFFUSION_DIR}/filtered_structures"
print("filtered_structures:", fs_dir)
print("num pdb:", len(glob.glob(f"{fs_dir}/*.pdb")))

In [ ]:
### Identifying loop regions in diffused backbones and setting up input JSON files for inpainting
import sys
PYTHON = {"general": "/home/machenshu/miniconda3/envs/diffusion/bin/python"}
print("Using python:", PYTHON["general"])
assert os.path.exists(f"{DIFFUSION_DIR}/filtered_structures"), "No diffused backbones to run inpainting on"
INPAINT_DIR = f"{WDIR}/inpainting/0_inpainting"
os.makedirs(INPAINT_DIR, exist_ok=True)
os.chdir(INPAINT_DIR)


inpaint_inputs_per_job = 2  # How many structures will go into each arrayed job
inpaint_catalytic_residues = "A15"  # Catalytic residues that were used as motif during diffusion, space-separated string

inpaint_setup_cmd = f"{PYTHON['general']} {SCRIPT_DIR}/scripts/diffusion_analysis/setup_inpaint_from_diffusion.py "\
               f"--group {inpaint_inputs_per_job} --var --design_full --params {' '.join(params)} "\
               f"--ref_catres {inpaint_catalytic_residues} "\
               f"--pdb {' '.join(glob.glob(f'{DIFFUSION_DIR}/filtered_structures/*.pdb'))}"

p = subprocess.Popen(inpaint_setup_cmd, shell=True)
(output, err) = p.communicate()


In [ ]:
### Setting up inpainting run commands (SAFE VERSION)

import os, glob, subprocess

# ---- 0) Make sure we use the PyRosetta-capable python (your diffusion env) ----
# You already set PYTHON["general"] = "/home/machenshu/miniconda3/envs/diffusion/bin/python"
# Here we unify diffusion runner to the same interpreter.
PYTHON["diffusion"] = PYTHON["general"]

# ---- 1) Must-have sanity checks to avoid path mistakes ----
# Make sure these variables exist in your notebook before this cell:
#   WDIR, SCRIPT_DIR, DIFFUSION_DIR, INPAINT_DIR, inpaint_script
assert os.path.exists(f"{DIFFUSION_DIR}/filtered_structures"), "No diffused backbones to run inpainting on"

# Your real inpaint.py path (must be absolute)
inpaint_script = "/home/machenshu/works/RFDesign/inpainting/inpaint.py"
assert os.path.exists(inpaint_script), f"inpaint_script not found: {inpaint_script}"

assert os.path.isdir(INPAINT_DIR), f"INPAINT_DIR not found: {INPAINT_DIR}"

# Find json files (non-recursive; if none found, try recursive)
jsons = sorted(glob.glob(f"{INPAINT_DIR}/*.json"))
if len(jsons) == 0:
    jsons = sorted(glob.glob(f"{INPAINT_DIR}/**/*.json", recursive=True))

print("PYTHON[general]  =", PYTHON["general"])
print("PYTHON[diffusion] =", PYTHON["diffusion"])
print("inpaint_script   =", inpaint_script)
print("INPAINT_DIR      =", INPAINT_DIR)
print("num json         =", len(jsons))
assert len(jsons) > 0, f"No json files found under {INPAINT_DIR}"

# ---- 2) Build commands and write commands_inpaint (for slurm array later) ----
commands_inpaint = []
cmds_filename = "commands_inpaint"
cmds_logfile = []

with open(cmds_filename, "w") as file:
    for jsn in jsons:
        cmd = [PYTHON["diffusion"], inpaint_script, f"--input_json={jsn}"]
        commands_inpaint.append(cmd)

        logf = jsn.replace(".json", ".log")
        cmds_logfile.append(logf)

        # Write one command per line (useful for sbatch array later)
        file.write(" ".join(cmd) + "\n")

print("\nExample inpainting command:")
print(" ".join(commands_inpaint[0]))
print(f"working directory: {os.getcwd()}")

# ---- 3) Run strategy: first run ONLY ONE job as a sanity check ----
# After the first one succeeds (log looks good), set test_only=False to run all (serially).
test_only = False   # <-- Change to False ONLY after 1st job succeeds

# If you use ".done" as a guard file, keep it. Otherwise this block runs.
if not os.path.exists(os.path.join(INPAINT_DIR, ".done")):
    run_ids = range(1) if test_only else range(len(commands_inpaint))

    for i in run_ids:
        print(f"\n>>> Running {i+1}/{len(commands_inpaint)}: {cmds_logfile[i]}")
        p = subprocess.Popen(commands_inpaint[i], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        output, err = p.communicate()

        with open(cmds_logfile[i], "w") as f:
            f.write(output)
            f.write("\n\n=== STDERR ===\n")
            f.write(err)

        print(f"Log written to: {cmds_logfile[i]}")
        if err.strip():
            print("STDERR:\n", err)
        else:
            print("STDERR: (empty) ✅")
        print("==================\n")
else:
    print("Found .done file — skipping inpainting execution.")


In [ ]:
## If you're done with inpainting and happy with the outputs then mark it as done
INPAINT_DIR = f"{WDIR}/inpainting/0_inpainting"
os.chdir(INPAINT_DIR)

if not os.path.exists(INPAINT_DIR+"/.done"):
    with open(f"{INPAINT_DIR}/.done", "w") as file:
        file.write(f"Run user: {username}\n")

### Aligning the ligand into inpainting outputs

In [ ]:
print("params type:", type(params))
print("params len:", len(params))
print("params sample:", params[:5])

In [ ]:
ref_pdbs = sorted(glob.glob(f"{DIFFUSION_DIR}/filtered_structures/*.pdb"))
inp_pdbs = sorted(glob.glob(f"{INPAINT_DIR}/*.pdb"))

print("num ref pdb      =", len(ref_pdbs))
print("num inpaint pdb  =", len(inp_pdbs))
print("estimated arg chars =", sum(len(x)+1 for x in ref_pdbs) + sum(len(x)+1 for x in inp_pdbs))


In [ ]:
import os, glob, subprocess, math

os.chdir(INPAINT_DIR)
os.makedirs("alignment", exist_ok=True)

ref_pdbs = sorted(glob.glob(f"{DIFFUSION_DIR}/filtered_structures/*.pdb"))
inp_pdbs = sorted(glob.glob(f"{INPAINT_DIR}/*.pdb"))

assert len(ref_pdbs) > 0
assert len(inp_pdbs) > 0

BATCH = 30
n_batches = math.ceil(len(inp_pdbs) / BATCH)

for bi in range(n_batches):
    batch_pdbs = inp_pdbs[bi*BATCH:(bi+1)*BATCH]

    cmd = (
        f"{PYTHON['general']} "
        f"{SCRIPT_DIR}/scripts/diffusion_analysis/align_dif_inpaint_add_ligand.py "
        f"--outdir alignment "
        f"--params {' '.join(params)} "
        f"--ref {' '.join(ref_pdbs)} "
        f"--pdb {' '.join(batch_pdbs)}"
    )

    print(f"\n=== Align batch {bi+1}/{n_batches}: {len(batch_pdbs)} structures ===")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = p.communicate()

    print("Return code:", p.returncode)
    if err:
        print("STDERR:\n", err[:1000])
    if p.returncode != 0:
        raise RuntimeError(f"Batch {bi+1} failed")


## 1: Running ProteinMPNN on inpainted backbones

We are first trying to just design a sequence on the backbone, without considering the ligand.
The goal is to first find backbones that fold well.

Alternatively, one could also go ahead and run the [LigandMPNN/FastRelax step](#ligmpnn_fr) already on the inpainted backbones.



=========================================================

**Feng's notes:** 11/27/2025

In order to run this step of the pipeline, I made a few changes as below:

+ Need to make a new conda env to run ligandMPNN. The original pipeline calling on to use the conda env named "general", which is identical to env "diffusion". You need to create a different one to run ligandMPNN, as specified in the ligandMPNN ReadMe page.

  We need to be carefully here to follow the ReadMe page in ligandMPNN, because there is a minor issue on it. I have revise it to fix that issue in the new repo of ligandMPNN.
  
+ When calling to run ligandMPNN, we need to specify the model weight file as indicated in the instruction. But the instruction is wrong about how. In the README of Heme_binder repo, it specifies that we need to modify the code in run_api.py file in LigandMPNN directory. It is wrong. After all, there is no such file. To specify the weight file, we need to specify the path to it as an argument to the calling. I have modify the code in the second cell below. You need to find out where you keep the model weight file and specify accordingly.

Again in the second cell below, find the line with 'f"--checkpoint_protein_mpnn {model_weight_dir}/LigandMPNN/model_params/proteinmpnn_v_48_020.pt\n")', and then change the path to your weight file.

+ Last modification to do has something to do with the matching file names for the .trb files and .pdf files in the code file run.py in LigandMPNN repo. The issue comes when the .trb files are using base name /short name, but the relevant pdf files are in absolute full path names. I modify the code already. The trick part is that the ligandMPNN is the submodule of the pipeline and we don't have permission to the original repo, therefore we have to create a new branch/copy to have the write permission. Please update the whole repo and make sure it points to the new repo (https://github.com/ffeng23/LigandMPNN_ff.git), not the original one.

After you finish the above changes, you should be able to run ligandMPNN!!!

====================


In [ ]:
import os, glob
WDIR = "/home/machenshu/works/heme_binder_diffusion_ff"
os.chdir(WDIR)
print("CWD =", os.getcwd())
print("pdb count =", len(glob.glob("inpainting/0_inpainting/*.pdb")))
print("trb count =", len(glob.glob("inpainting/0_inpainting/*.trb")))

In [ ]:
INPAINT_DIR = f"{WDIR}/inpainting/0_inpainting"

# 你的 pdb 就在 0_inpainting 根目录，不在 alignment/
pdb_files = sorted(glob.glob(INPAINT_DIR + "/*.pdb"))
assert len(pdb_files) > 0, f"No good backbones found in {INPAINT_DIR}!"

num_structures = len(pdb_files)
print(f"*** found {num_structures} structures in the directory!!")
print("Example pdb:", pdb_files[0])

os.chdir(WDIR)

MPNN_DIR = f"{WDIR}/inpainting/1_proteinmpnn"
os.makedirs(MPNN_DIR, exist_ok=True)
os.chdir(MPNN_DIR)

print("Start writing json file for next step.............")

# trb 也大概率在根目录；如果不在，就再补一个 alignment/ 兜底
trb_files = sorted(glob.glob(INPAINT_DIR + "/*.trb"))
if len(trb_files) == 0:
    trb_files = sorted(glob.glob(INPAINT_DIR + "/alignment/*.trb"))

assert len(trb_files) > 0, f"No .trb files found in {INPAINT_DIR} (or alignment/)"

mask_json_cmd = (
    f"{PYTHON['general']} {SCRIPT_DIR}/scripts/design/make_maskdict_from_trb.py "
    f"--out masked_pos.jsonl --trb {' '.join(trb_files)}"
)

result = subprocess.run(mask_json_cmd, shell=True, text=True, capture_output=True)
print("----stdout:\n", result.stdout)
print("----stderr:\n", result.stderr)
print("----return code:", result.returncode)

assert os.path.exists("masked_pos.jsonl"), "Failed to create masked positions JSONL file"
print("\n*** Successfully wrote masked_pos.jsonl for running MPNN *****\n")


Split original code into two parts for easy debugging (Feng)

In [ ]:
# =========================
# Final Cell: Run ProteinMPNN (LigandMPNN repo)
# =========================
import os, glob, subprocess

# ---- 0) Basic paths must already exist in your notebook ----
# Required: WDIR, SCRIPT_DIR, INPAINT_DIR, MPNN_DIR, proteinMPNN_script, PYTHON (dict)
# If any of these are not defined earlier, you need to define them in prior cells.

# ---- 1) MPNN settings ----
MPNN_temperatures = [0.1, 0.2, 0.3]       # e.g. [0.1, 0.2, 0.3] for full run
commands_by_T = {0.1: [], 0.2: [], 0.3: []}

for cmd in commands_mpnn:
    for T in commands_by_T:
        if f"--temperature {T}" in cmd:
            commands_by_T[T].append(cmd)

for T in commands_by_T:
    fname = f"commands_mpnn_T{T}"
    with open(fname, "w") as f:
        for c in commands_by_T[T]:
            f.write(c + "\n")
    print(T, ":", len(commands_by_T[T]))
MPNN_outputs_per_temperature = 5   # number_of_batches in LigandMPNN/run.py
MPNN_omit_AAs = "CM"               # omit Cys/Met if needed

# ---- 2) Locate input PDBs (yours are in 0_inpainting root, NOT alignment/) ----
pdb_list = sorted(glob.glob(os.path.join(INPAINT_DIR, "*.pdb")))
assert len(pdb_list) > 0, f"No pdb found in {INPAINT_DIR}"
print("Total pdb backbones:", len(pdb_list))
print("Example pdb:", pdb_list[0])

# ---- 3) Python path (ligandmpnn conda env) ----
PYTHON["ligandmpnn"] = "/home/machenshu/miniconda3/envs/ligandmpnn/bin/python"
assert os.path.exists(PYTHON["ligandmpnn"]), f"Python not found: {PYTHON['ligandmpnn']}"

# ---- 4) Checkpoint path (use repo internal weights) ----
CHECKPOINT = os.path.join(WDIR, "lib/LigandMPNN/model_params/proteinmpnn_v_48_020.pt")
assert os.path.exists(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"
print("Using checkpoint:", CHECKPOINT)

# ---- 5) LigandMPNN script path (run.py) ----
# proteinMPNN_script should point to .../lib/LigandMPNN/run.py
if not os.path.isabs(proteinMPNN_script):
    proteinMPNN_script = os.path.join(WDIR, proteinMPNN_script)
assert os.path.exists(proteinMPNN_script), f"proteinMPNN_script not found: {proteinMPNN_script}"
print("proteinMPNN_script:", proteinMPNN_script)

# ---- 6) Ensure output directory exists ----
os.makedirs(MPNN_DIR, exist_ok=True)
os.chdir(MPNN_DIR)
print("CWD:", os.getcwd())

# ---- 7) OPTIONAL: clean previous smoke-test outputs to avoid confusion/overwrite ----
# Comment out if you want to keep old files.
for fp in glob.glob("seqs/*.fa") + glob.glob("backbones/*.pdb"):
    try:
        os.remove(fp)
    except:
        pass

# ---- 8) Write commands file ----
commands_mpnn = []
cmds_filename_mpnn = "commands_mpnn"

with open(cmds_filename_mpnn, "w") as file:
    for T in MPNN_temperatures:
        for pdb_path in pdb_list:
            cmd = (
                f"{PYTHON['ligandmpnn']} {proteinMPNN_script} "
                f"--model_type protein_mpnn "
                f"--ligand_mpnn_use_atom_context 0 "
                f"--fixed_residues_multi masked_pos.jsonl "
                f"--out_folder ./ "
                f"--number_of_batches {MPNN_outputs_per_temperature} "
                f"--temperature {T} "
                f"--omit_AA {MPNN_omit_AAs} "
                f"--pdb_path {pdb_path} "
                f"--checkpoint_protein_mpnn {CHECKPOINT}"
            )
            commands_mpnn.append(cmd)
            file.write(cmd + "\n")

print(f"{len(commands_mpnn)} MPNN jobs to run")
print("Example MPNN command:\n", commands_mpnn[-1])

# ---- 9) Smoke test: run first 2 commands ----
# NOTE: do NOT use assert until you've printed stderr; use RuntimeError instead for clearer logs.
if os.path.exists(os.path.join(MPNN_DIR, ".done")):
    print("Found .done -> skip running. Remove .done if you want to rerun.")
else:
    for i, cmd_i in enumerate(commands_mpnn[:2], start=1):
        print(f"\nRun {i}/{len(commands_mpnn)} designing...")
        p = subprocess.run(cmd_i, shell=True, text=True, capture_output=True)

        print("return code:", p.returncode)
        if p.stdout:
            print("\n===== STDOUT (head) =====\n", p.stdout[:2000])
        if p.stderr:
            print("\n===== STDERR (head) =====\n", p.stderr[:2000])

        if p.returncode != 0:
            raise RuntimeError("MPNN command failed (see stdout/stderr above)")

print("\n✅ Smoke test finished. Check outputs:")
print(" -", os.path.join(MPNN_DIR, "seqs"))
print(" -", os.path.join(MPNN_DIR, "backbones"))


In [ ]:
## If you're done with MPNN and happy with the outputs then mark it as done
MPNN_DIR = f"{WDIR}/inpainting/1_proteinmpnn"
os.chdir(MPNN_DIR)

if not os.path.exists(MPNN_DIR+"/.done"):
    with open(f"{MPNN_DIR}/.done", "w") as file:
        file.write(f"Run user: {username}\n")


## 2: Running AlphaFold2
Performing AF2 single sequence predictions

**Feng's Notes:**
To run alphafold, we need to make the following changes:

+ change the model weights/params directory. I have downloaded the model parametes in a different folder. So, we need to change the path to the parameters according. The file to be changed is in "scripts/af2/AlphaFold2.py [line 40]"

+ need to change the "mlfold" conda environment to install nvcc

+ the alphafold code require a newer version of python (>3.11), which allow for "@typing.dataclass_transform decorator" in alphafold/model/base_config.py file. The mlfold evn contains an older Python version (3.10), we must use the typing_extensions library to access this feature.

      conda install -c conda-forge typing-extensions

And also import at the file alphafold/model/base_config.py

        try:
            from typing import dataclass_transform
        except ImportError:
            from typing_extensions import dataclass_transform

Lastly, change the statement
        @type.dataclass_transform()

to the statement
        @dataclass_transform()
        
+ jax compatibility with cudatoolkit. jaxlib library version has to match with cudatoolkit library version.

        jaxlib.xla_extension.XlaRuntimeError: FAILED_PRECONDITION: Couldn't get ptxas/nvlink version string: INTERNAL: Couldn't invoke ptxas --version

This error indicates that JAX cannot find or execute the NVIDIA CUDA compiler components (ptxas and nvlink) needed for GPU operation. For us, the jax version in mlfold env is too low and doesn't match the cudatoolkit version. To rescue, we need to first update cudatoolkit version (conda update -c conda-forge cudatoolkit); and second, install nvcc (conda install -c nvidia cuda-nvcc). 

After these two step, we still need to run the jupyter lab under the mlfold env.

    conda activate mlfold
    jupyter lab --no-browser --port=8888

You also need to install jupyterlab under mlfold if this has not been done.

+ Change the script scripts/af2/af2.py at line 42 to save the file with the overwriten mode
  
      with open(args.scorefile, "w") as file:

This was originally in the "append" mode ("a"). This might break the downstream code when we try to run the script multiple times.



In [9]:
import os, glob

ROOT = "/home/machenshu/works/heme_binder_diffusion_ff"
WDIR = ROOT

# 三个温度对应的 seqs 目录（来自你截图的前三个）
T01_DIR = f"{ROOT}/inpainting/1_proteinmpnn/T0.1/seqs"
T02_DIR = f"{ROOT}/inpainting/1_proteinmpnn/T0.2/seqs"
T03_DIR = f"{ROOT}/inpainting/1_proteinmpnn/results_T0.3/seqs"

for d in [T01_DIR, T02_DIR, T03_DIR]:
    print(d, "exists?", os.path.isdir(d), "fa:", len(glob.glob(d+"/*.fa")))

AF2_DIR = f"{WDIR}/inpainting/2_af2"
AF2_INPUT_DIR = f"{AF2_DIR}/af2_input_fasta"
os.makedirs(AF2_INPUT_DIR, exist_ok=True)
print("AF2_INPUT_DIR:", AF2_INPUT_DIR)


/home/machenshu/works/heme_binder_diffusion_ff/inpainting/1_proteinmpnn/T0.1/seqs exists? True fa: 179
/home/machenshu/works/heme_binder_diffusion_ff/inpainting/1_proteinmpnn/T0.2/seqs exists? True fa: 179
/home/machenshu/works/heme_binder_diffusion_ff/inpainting/1_proteinmpnn/results_T0.3/seqs exists? True fa: 179
AF2_INPUT_DIR: /home/machenshu/works/heme_binder_diffusion_ff/inpainting/2_af2/af2_input_fasta


In [10]:
import os, glob

src_dirs = [
    ("T0.1", T01_DIR),
    ("T0.2", T02_DIR),
    ("T0.3", T03_DIR),
]

linked = 0
skipped = 0

for tag, d in src_dirs:
    fa_files = glob.glob(d + "/*.fa")
    for f in fa_files:
        dst = os.path.join(AF2_INPUT_DIR, f"{tag}__{os.path.basename(f)}")
        if os.path.exists(dst):
            skipped += 1
            continue
        os.symlink(f, dst)
        linked += 1

print("linked:", linked, "skipped(existing):", skipped)
print("total fasta in AF2_INPUT_DIR:", len(glob.glob(AF2_INPUT_DIR + "/*.fa")))


linked: 537 skipped(existing): 0
total fasta in AF2_INPUT_DIR: 537


In [11]:
import glob

fa_files = glob.glob(AF2_INPUT_DIR + "/*.fa")
n_files = len(fa_files)

n_seqs = 0
for f in fa_files:
    with open(f) as fh:
        for line in fh:
            if line.startswith(">"):
                n_seqs += 1

print("FA files:", n_files)
print("Total sequences (headers):", n_seqs)


FA files: 537
Total sequences (headers): 3222


In [13]:
import os, sys, glob

ROOT = "/home/machenshu/works/heme_binder_diffusion_ff"
UTILS_DIR = ROOT + "/scripts/utils"
if UTILS_DIR not in sys.path:
    sys.path.append(UTILS_DIR)

import utils
print("utils loaded from:", utils.__file__)


utils loaded from: /home/machenshu/works/heme_binder_diffusion_ff/scripts/utils/utils.py


In [14]:
os.chdir(WDIR)
assert len(glob.glob(MPNN_DIR+"/seqs/*.fa")) > 0, "No MPNN outputs to run AF2 on"

AF2_DIR = f"{WDIR}/inpainting/2_af2"
os.makedirs(AF2_DIR, exist_ok=True)
os.chdir(AF2_DIR)

### First collecting MPNN outputs and creating FASTA files for AF2 input
mpnn_fasta = utils.parse_fasta_files(glob.glob(f"{AF2_INPUT_DIR}/*.fa"))
#print(mpnn_fasta)
mpnn_fasta = {k: seq.strip() for k, seq in mpnn_fasta.items() if "model_path" not in k}  # excluding the diffused poly-A sequence

#print(mpnn_fasta)
# Giving sequences unique names based on input PDB name, temperature, and sequence identifier
mpnn_fasta = {k.split(",")[0]+"_"+k.split(",")[2].replace(" T=", "T")+"_0_"+k.split(",")[1].replace(" id=", ""): seq for k, seq in mpnn_fasta.items()}

print(f"A total on {len(mpnn_fasta)} sequences will be predicted.")

## Splitting the MPNN sequences based on length
## and grouping them in smaller batches for each AF2 job
## Use group size of >40 when running on GPU. Also depends on how many sequences and resources you have.
SEQUENCES_PER_AF2_JOB = 4  # CPU
if USE_GPU_for_AF2 is True:
    SEQUENCES_PER_AF2_JOB = 100  # GPU
mpnn_fasta_split = utils.split_fasta_based_on_length(mpnn_fasta, SEQUENCES_PER_AF2_JOB, write_files=True)


A total on 2685 sequences will be predicted.
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_2_T0.2_0_2
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_2_T0.2_0_3
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_2_T0.2_0_4
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_2_T0.2_0_5
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_1_T0.2_0_2
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_1_T0.2_0_3
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_1_T0.2_0_4
Duplicate sequence: >7o2g_HBA_dif_29_traj1_inp_1_T0.2_0_5
Duplicate sequence: >7o2g_HBA_dif_62_traj3_inp_4_T0.2_0_2
Duplicate sequence: >7o2g_HBA_dif_62_traj3_inp_4_T0.2_0_3
Duplicate sequence: >7o2g_HBA_dif_62_traj3_inp_4_T0.2_0_4
Duplicate sequence: >7o2g_HBA_dif_62_traj3_inp_4_T0.2_0_5
Duplicate sequence: >7o2g_HBA_dif_51_inp_4_T0.2_0_2
Duplicate sequence: >7o2g_HBA_dif_51_inp_4_T0.2_0_3
Duplicate sequence: >7o2g_HBA_dif_51_inp_4_T0.2_0_4
Duplicate sequence: >7o2g_HBA_dif_51_inp_4_T0.2_0_5
Duplicate sequence: >7o2g_HBA_dif_85_traj1_inp_1_T0

In [20]:
import os, glob, subprocess, sys
AF2_script = "/home/machenshu/works/heme_binder_diffusion_ff/scripts/af2/af2.py"
AF2_recycles = 3
AF2_models = "4"

# 1) 找到 fasta
fasta_files = sorted(glob.glob("*.fasta"))
print("FASTA files:", len(fasta_files))
print("example:", fasta_files[:3])
assert len(fasta_files) > 0, "No *.fasta files found in current AF2_DIR"

# 2) 直接用当前 kernel 的 python（你现在是 mlfold）
af2_python = sys.executable
print("af2_python:", af2_python)

print("AF2_script:", AF2_script)
assert os.path.exists(AF2_script), "AF2_script path invalid"

# 3) 断点续跑：如果 csv 已存在就跳过
commands_af2 = []
cmds_filename_af2 = "commands_af2"

with open(cmds_filename_af2, "w") as file:
    for ff in fasta_files:
        score = ff.replace(".fasta", ".csv")
        if os.path.exists(score) and os.path.getsize(score) > 0:
            continue
        cmd = (
            f"{af2_python} {AF2_script} "
            f"--af-nrecycles {AF2_recycles} --af-models {AF2_models} "
            f"--fasta {ff} --scorefile {score}"
        )
        commands_af2.append(cmd)
        file.write(cmd + "\n")

print("Commands to run:", len(commands_af2))
print("Example AF2 command:\n", commands_af2[-1] if commands_af2 else "None (all done)")

if len(commands_af2) == 0:
    print("All AF2 jobs already completed (csv exists).")
else:
    submit_script = "submit_af2.sh"
    if USE_GPU_for_AF2:
        utils.create_slurm_submit_script(
    filename=submit_script, name="2_af2", mem="16g",
    N_cores=4, gpu=True, gres="gpu:1", time="01:00:00",
    array=len(commands_af2), array_commandfile=cmds_filename_af2
    )
    else:
        utils.create_slurm_submit_script(
            filename=submit_script, name="2_af2", mem="16g",
            N_cores=8, time="02:00:00",
            array=len(commands_af2), array_commandfile=cmds_filename_af2
        )

    print("Submitting:", submit_script)
    print("Running AF2 locally...")

count = 0
for cmd in commands_af2:
    count += 1
    print(f"\n=== Run {count}/{len(commands_af2)} ===")
    print(cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    out, err = p.communicate()
    print("STDOUT:\n", out)
    print("STDERR:\n", err)

FASTA files: 29
example: ['104aa_0.fasta', '105aa_0.fasta', '106aa_0.fasta']
af2_python: /home/machenshu/miniconda3/envs/mlfold/bin/python
AF2_script: /home/machenshu/works/heme_binder_diffusion_ff/scripts/af2/af2.py
Commands to run: 29
Example AF2 command:
 /home/machenshu/miniconda3/envs/mlfold/bin/python /home/machenshu/works/heme_binder_diffusion_ff/scripts/af2/af2.py --af-nrecycles 3 --af-models 4 --fasta 140aa_0.fasta --scorefile 140aa_0.csv
Submitting: submit_af2.sh
Running AF2 locally...

=== Run 1/29 ===
/home/machenshu/miniconda3/envs/mlfold/bin/python /home/machenshu/works/heme_binder_diffusion_ff/scripts/af2/af2.py --af-nrecycles 3 --af-models 4 --fasta 104aa_0.fasta --scorefile 104aa_0.csv
STDOUT:
 Setting up models took 2.285 seconds.
/tmp/xrh0mglk
Sequence 0 completed in 40.0 sec with 1 models; lDDT=57.415
Sequence 1 completed in 2.1 sec with 1 models; lDDT=70.527
Sequence 2 completed in 2.1 sec with 1 models; lDDT=79.249
Sequence 3 completed in 2.1 sec with 1 models; lD

In [ ]:
## If you're done with AF2 and happy with the outputs then mark it as done
AF2_DIR = f"{WDIR}/inpainting/2_af2"
os.chdir(AF2_DIR)

if not os.path.exists(AF2_DIR+"/.done"):
    with open(f"{AF2_DIR}/.done", "w") as file:
        file.write(f"Run user: {username}\n")


### Analyzing AlphaFold2 predictions

In [ ]:
INPAINT_DIR = f"{WDIR}/inpainting/0_inpainting"

# Combining all CSV scorefiles into one
os.system("head -n 1 $(ls *aa*.csv | shuf -n 1) > scores.csv ; for f in *aa*.csv ; do tail -n +2 ${f} >> scores.csv ; done")
assert os.path.exists("scores.csv"), "Could not combine scorefiles"

### Calculating the RMSDs of AF2 predictions relative to the diffusion outputs
### Catalytic residue sidechain RMSDs are calculated in the reference PDB has REMARK 666 line present

analysis_cmd = f"{PYTHON['general']} {SCRIPT_DIR}/scripts/utils/analyze_af2.py --scorefile scores.csv "\
               f"--ref_path {INPAINT_DIR}/alignment/ --mpnn --params {' '.join(params)}"
print(analysis_cmd)

p = subprocess.Popen(analysis_cmd, shell=True)
(output, err) = p.communicate()
print(f"err:{err}")
print(f"output:{output}")

In [ ]:
### Visualizing and filtering AF2 predictions
AF2_DIR = f"{WDIR}/inpainting/2_af2"

scores_af2 = pd.read_csv("scores.sc", sep="\s+", header=0)

### Filtering AF2 scores based on lddt and rmsd
# Define your desired cutoffs here:
AF2_filters = {"lDDT": [85.0, ">="],
               "rmsd": [1.5, "<="],
               "rmsd_SR1": [2.0, "<="]}  # 1st catalytic residue sc-rmsd

scores_af2_filtered = utils.filter_scores(scores_af2, AF2_filters)
utils.dump_scorefile(scores_af2_filtered, "filtered_scores.sc")

## Plotting AF2 scores
plt.figure(figsize=(12, 3))
for i,k in enumerate(AF2_filters):
    plt.subplot(1, 3, i+1)
    plt.hist(scores_af2[k])
    plt.title(k)
    plt.xlabel(k)
plt.tight_layout()
plt.show()
    
utils.plot_score_pairs(scores_af2, "lDDT", "rmsd", AF2_filters["lDDT"][0], AF2_filters["rmsd"][0])

In [ ]:
### Copying good predictions to a separate directory
os.chdir(AF2_DIR)
INPAINT_DIR = f"{WDIR}/inpainting/0_inpainting"

if len(scores_af2_filtered) > 0:
    os.makedirs("good", exist_ok=True)
    good_af2_models = [row["Output_PDB"]+".pdb" for idx,row in scores_af2_filtered.iterrows()]
    for pdb in good_af2_models:
        copy2(pdb, f"good/{pdb}")
    good_af2_models = glob.glob(f"{AF2_DIR}/good/*.pdb")
else:
    sys.exit("No good models to continue this pipeline with")

os.chdir(f"{AF2_DIR}/good")


### Aligning the ligand back into the AF2 predictions.
### This is done by aligning the AF2 model to diffusion output and copying over the ligand using PyRosetta.
### --fix_catres option will re-adjust the rotamer and tautomer of 
### any catalytic residue to be the same as in the reference model.

align_cmd = f"{PYTHON['general']} {SCRIPT_DIR}/scripts/utils/place_ligand_after_af2.py "\
            f"--outdir with_heme --params {' '.join(params)} --fix_catres "\
            f"--pdb {' '.join(good_af2_models)} "\
            f"--ref {' '.join(glob.glob(INPAINT_DIR+'/alignment/*.pdb'))}"

p = subprocess.Popen(align_cmd, shell=True)
(output, err) = p.communicate()

## 3.1: Performing binding site design with LigandMPNN / FastRelax
<a id='ligmpnn_fr'></a>

**Feng's notes**:
Unfortunately, in this section, there is a missing package defining "mpnn_api". It seems that this is a component of the old pipeline, because in the README it mentions it as "mpnn_api.py". It has totally different api as to the "new" ProteinMPNN class. We surely could switch the code to make use of the "new" ProteinMPNN api, but it takes time to switch and debug. We might also ask for help from the authors of the pipeline, and it takes time too.
Currently, we just skip this section and continue to the following step. If necessary, we can come back later.

Another change to this section is to add "libtiff.so.5". This shared library is needed by PIL (Pillow?) in the code, but since Ubuntu 24.04, this has been updated to libtiff.so.6. We can manually install libtiff5 by downloading tiff5 deb and install that way. Another easy trick is to symbolically link to tiff6. 

        cd /usr/lib/x86_64-linux-gnu/
        ln -s libtiff.so.6 libtiff.so.5

To-do:
we might need to rewrite the mpnn_api to make this part working. Another temporary fix could be
+ split the run of heme_pocket_ligMPNN into two parts,
+ in the first part, we figure out using the code in the script the residues to be designed.
+ with these residues we call ligandMPNN to do redesign, without side-chain packing, with consideration of constraints(?).

(Side nodes: Protein side-chain packing (PSCP) involves predicting the three-dimensional coordinates of a protein's side-chain atoms given the backbone)

updated 12/16/2025 



In [ ]:
################################################
#------- DO NOT RUN This section-------------
#####################################################


### Setting up design directory and commands
os.chdir(WDIR)
DESIGN_DIR_ligMPNN = f"{WDIR}/inpainting/3.1_design_pocket_ligandMPNN"
os.makedirs(DESIGN_DIR_ligMPNN, exist_ok=True)
os.chdir(DESIGN_DIR_ligMPNN)

AF2_DIR = f"{WDIR}/inpainting/2_af2"  # change this if you want to run it on inpainting outputs
os.makedirs(DESIGN_DIR_ligMPNN+"/logs", exist_ok=True)

### Performing 5 design iterations on each input structure
NSTRUCT = 10
cstfile = f"{SCRIPT_DIR}/theozyme/HBA/HBA_CYS_UPO.cst"

commands_design = []
cmds_filename_des = "commands_design"
with open(cmds_filename_des, "w") as file:
    for pdb in glob.glob(f"{AF2_DIR}/good/with_heme/*.pdb"):
        commands_design.append(f"{PYTHON['general']} {SCRIPT_DIR}/scripts/design/heme_pocket_ligMPNN.py "
                             f"--pdb {pdb} --nstruct {NSTRUCT} "
                             f"--scoring {SCRIPT_DIR}/scripts/design/scoring/heme_scoring.py "
                             f"--params {' '.join(params)} --cstfile {cstfile} > logs/{os.path.basename(pdb).replace('.pdb', '.log')}\n")
        file.write(commands_design[-1])

print("Example design command:")
print(commands_design[-1])


### Running design jobs with Slurm.
#submit_script = "submit_design.sh"
#utils.create_slurm_submit_script(filename=submit_script, name="3.1_design_pocket_ligMPNN", mem="4g", 
#                                 N_cores=1, time="3:00:00", email=EMAIL, array=len(commands_design),
#                                 array_commandfile=cmds_filename_des)
print(os.getcwd())
if not os.path.exists(DESIGN_DIR_ligMPNN+"/.done"):
    for cmd_i in commands_design[0:1]:
        p = subprocess.Popen(cmd_i, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
        (output, err) = p.communicate()
        print("err:\n",err)

In [ ]:
################################################
#------- DO NOT RUN This section-------------
#####################################################
DESIGN_DIR_ligMPNN = f"{WDIR}/inpainting/3.1_design_pocket_ligandMPNN"
os.chdir(DESIGN_DIR_ligMPNN)

## If you're done with design and happy with the outputs then mark it as done
if not os.path.exists(DESIGN_DIR_ligMPNN+"/.done"):
    with open(f"{DESIGN_DIR_ligMPNN}/.done", "w") as file:
        file.write(f"Run user: {username}\n")

In [ ]:

################################################
#------- DO NOT RUN This section-------------
#####################################################


## Analyzing Rosetta designs
scores = pd.read_csv("scorefile.txt", sep="\s+", header=0)

filters = {'all_cst': [1.0, '<='],
 'nlr_SR1_rms': [0.8, '<='],
 'nlr_totrms': [1.0, '<='],
 'L_SASA': [0.2, '<='],
 'COO_hbond': [1.0, '='],
 'heme_angle_wrst': [80.0, '>='],
 'score_per_res': [0.0, '<='],
 'corrected_ddg': [-50.0, '<='],
 'cms_per_atom': [4.8, '>=']}

filtered_scores = utils.filter_scores(scores, filters)

## Plotting design scores
plt.figure(figsize=(12, 9))
for i,k in enumerate(filters):
    plt.subplot(3, 3, i+1)
    plt.hist(scores[k])
    plt.title(k)
    plt.xlabel(k)
plt.tight_layout()
plt.show()

### Copying good designs over to a new directory
if len(filtered_scores) > 0:
    os.makedirs(f"{DESIGN_DIR_ligMPNN}/good", exist_ok=True)
    for idx, row in filtered_scores.iterrows():
        copy2(row["description"]+".pdb", "good/"+row["description"]+".pdb")
else:
    print("No good designs created, bummer...")

So far, we skip the heme pocket ligMPNN design, continue on with the next section.!!

## 4.1 Performing ligandMPNN redesign on the 2nd layer residues

Resampling residues that are not in the pocket, but also not very far from the pocket

**Feng's notes:**

Since we skipped the last section, we need to copy over the good designs from the previous section. 

+ make a new subfold called "good" under "3.1_design_pocket_ligandMPNN"
+ copy all pdf files from "2_af2/good/with_heme" to "3.1_design_pocket_ligandMPNN/good/"

Then continue to run the code below.

+ also remember to change the path to the model parameter file when calling ligandmpnn script (check the comments below.
  

In [ ]:
if len(glob.glob(DESIGN_DIR_ligMPNN+'/good/*.pdb')) == 0:
    sys.exit("No designs to run 2nd MPNN on.")

os.chdir(WDIR)
DESIGN_DIR_2nd_mpnn = f"{WDIR}/inpainting/4.1_2nd_mpnn"
os.makedirs(DESIGN_DIR_2nd_mpnn, exist_ok=True)
os.chdir(DESIGN_DIR_2nd_mpnn)

### Making a JSON file specifiying designable positions for each structure.
### Will also make non-pocket ALA positions as designable.
### This is to fix any surface ALA-patches that previous MPNN may have introduced.

make_json_cmd = f"{PYTHON['general']} {SCRIPT_DIR}/scripts/design/setup_ligand_mpnn_2nd_layer.py "\
                f"--params {' '.join(params)} --ligand {LIGAND} --output_path parsed_pdbs_lig.jsonl "\
                 "--output_path masked_pos.jsonl --dist_bb 6.0 --dist_sc 5.0 "\
                f"--pdb {' '.join(glob.glob(DESIGN_DIR_ligMPNN+'/good/*.pdb'))}"

p = subprocess.Popen(make_json_cmd, shell=True)
(output, err) = p.communicate()

print(f"err:{err}")

if not os.path.exists("masked_pos.jsonl"):
    sys.exit()

### Setting up ligandMPNN run commands
## We're doing design with 2 temperatures (more conservative than before), and 5 sequences each.

MPNN_temperatures = [0.1, 0.2]
MPNN_outputs_per_temperature = 5
MPNN_omit_AAs = "CM"

commands_mpnn = []
cmds_filename_mpnn = "commands_mpnn"

#############################################
#
# Here remember to change the path to the param file,ligandmpnn_v_32_010_25.pt
#
###############################################
with open(cmds_filename_mpnn, "w") as file:
    for T in MPNN_temperatures:
        for f in glob.glob(DESIGN_DIR_ligMPNN+'/good/*.pdb'):
            commands_mpnn.append(f"{PYTHON['ligandMPNN']} {proteinMPNN_script} "
                                 f"--model_type ligand_mpnn --ligand_mpnn_use_atom_context 1 "
                                 "--fixed_residues_multi masked_pos.jsonl --out_folder ./ "
                                 f"--number_of_batches {MPNN_outputs_per_temperature} --temperature {T} "
                                 f"--omit_AA {MPNN_omit_AAs} --pdb_path {f} "
                                 f"--checkpoint_ligand_mpnn /mnt/data/heme_rfdiffusion_data/LigandMPNN/model_params/ligandmpnn_v_32_010_25.pt\n")
            file.write(commands_mpnn[-1])

print("Example MPNN command:")
print(commands_mpnn[-1])

### Running ligandMPNN with Slurm.
### Grouping jobs with 10 commands per one array job.

submit_script = "submit_mpnn.sh"
utils.create_slurm_submit_script(filename=submit_script, name="4.1_2nd_mpnn", mem="8g", 
                                 N_cores=1, time="0:15:00", email=EMAIL, array=len(commands_mpnn),
                                 array_commandfile=cmds_filename_mpnn, group=10)

if not os.path.exists(DESIGN_DIR_2nd_mpnn+"/.done"):
    for command_i in commands_mpnn:
        p = subprocess.Popen(command_i, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
        (output, err) = p.communicate()
        print(f"err:{err}")


In [ ]:
DESIGN_DIR_ligMPNN = f"{WDIR}/inpainting/3.1_design_pocket_ligandMPNN"
DESIGN_DIR_2nd_mpnn = f"{WDIR}/inpainting/4.1_2nd_mpnn"
os.chdir(DESIGN_DIR_2nd_mpnn)
if len(glob.glob(DESIGN_DIR_ligMPNN+'/good/*.pdb')) == 0:
    sys.exit("No designs to run 2nd MPNN on.")

## If you're done with design and happy with the outputs then mark it as done
if not os.path.exists(DESIGN_DIR_2nd_mpnn+"/.done"):
    with open(f"{DESIGN_DIR_2nd_mpnn}/.done", "w") as file:
        file.write(f"Run user: {username}\n")

## 5.1 AlphaFold2 predictions on the 2nd MPNN run

In [ ]:
os.chdir(WDIR)
assert os.path.exists(DESIGN_DIR_2nd_mpnn+"/.done"), "2nd MPNN has not been performed!"

AF2_DIR = f"{WDIR}/inpainting/5.1_2nd_af2"
os.makedirs(AF2_DIR, exist_ok=True)
os.chdir(AF2_DIR)

### First collecting MPNN outputs and creating FASTA files for AF2 input
mpnn_fasta = utils.parse_fasta_files(glob.glob(f"{DESIGN_DIR_2nd_mpnn}/seqs/*.fa"))
# Giving sequences unique names based on input PDB name, temperature, and sequence identifier
_mpnn_fasta = {}
for k, seq in mpnn_fasta.items():
    if "model_path" in k:
        _mpnn_fasta[k.split(",")[0]+"_native"] = seq.strip()
    else:
        _mpnn_fasta[k.split(",")[0]+"_"+k.split(",")[2].replace(" T=", "T")+"_0_"+k.split(",")[1].replace(" id=", "")] = seq.strip()
mpnn_fasta = {k:v for k,v in _mpnn_fasta.items()}

print(f"A total on {len(mpnn_fasta)} sequences will be predicted.")

## Splitting the MPNN sequences based on length
## and grouping them in smaller batches for each AF2 job
## Use group size of >40 when running on GPU. Also depends on how many sequences and resources you have.
SEQUENCES_PER_AF2_JOB = 5  # CPU
if USE_GPU_for_AF2 is True:
    SEQUENCES_PER_AF2_JOB = 100  # GPU
mpnn_fasta_split = utils.split_fasta_based_on_length(mpnn_fasta, SEQUENCES_PER_AF2_JOB, write_files=True)


In [ ]:
## Setting up AlphaFold2 run

AF2_recycles = 3
AF2_models = "4"  # add other models to this string if needed

commands_af2 = []
cmds_filename_af2 = "commands_af2"
with open(cmds_filename_af2, "w") as file:
    for ff in glob.glob("*.fasta"):
        commands_af2.append(f"{PYTHON['af2']} {AF2_script} "
                             f"--af-nrecycles {AF2_recycles} --af-models {AF2_models} "
                             f"--fasta {ff} --scorefile {ff.replace('.fasta', '.csv')}\n")
        file.write(commands_af2[-1])

print("Example AF2 command:")
print(commands_af2[-1])

print(f"# of commands to run:{len(commands_af2)}")

### Running AF2 with Slurm.
### Running jobs on the CPU. It takes ~10 minutes per sequence

submit_script = "submit_af2.sh"
if USE_GPU_for_AF2 is True:
    utils.create_slurm_submit_script(filename=submit_script, name="5.1_2nd_af2", mem="6g", 
                                     N_cores=2, gpu=True, gres="gpu:rtx2080:1", time="00:12:00", email=EMAIL, array=len(commands_af2),
                                     array_commandfile=cmds_filename_af2)
else:
    utils.create_slurm_submit_script(filename=submit_script, name="5.1_2nd_af2", mem="6g", 
                                     N_cores=4, time="01:00:00", email=EMAIL, array=len(commands_af2),
                                     array_commandfile=cmds_filename_af2)

print(os.getcwd())
if not os.path.exists(AF2_DIR+"/.done"):
    for command_i in commands_af2:
        #print(command_i)
        p = subprocess.Popen(command_i, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
        (output, err) = p.communicate()
        print(f"err:\n{err}")
        


In [ ]:
## If you're done with AF2 and happy with the outputs then mark it as done
AF2_DIR = f"{WDIR}/inpainting/5.1_2nd_af2"
DESIGN_DIR_ligMPNN = f"{WDIR}/inpainting/3.1_design_pocket_ligandMPNN"
os.chdir(AF2_DIR)

if not os.path.exists(AF2_DIR+"/.done"):
    with open(f"{AF2_DIR}/.done", "w") as file:
        file.write(f"Run user: {username}\n")

### Analyzing AlphaFold2 predictions

In [ ]:
# Combining all CSV scorefiles into one
os.system("head -n 1 $(ls *aa*.csv | shuf -n 1) > scores.csv ; for f in *aa*.csv ; do tail -n +2 ${f} >> scores.csv ; done")
assert os.path.exists("scores.csv"), "Could not combine scorefiles"

### Calculating the RMSDs of AF2 predictions relative to the diffusion outputs
### Catalytic residue sidechain RMSDs are calculated in the reference PDB has REMARK 666 line present

analysis_cmd = f"{PYTHON['general']} {SCRIPT_DIR}/scripts/utils/analyze_af2.py --scorefile scores.csv "\
               f"--ref_path {DESIGN_DIR_ligMPNN}/good/ --mpnn --params {' '.join(params)}"

p = subprocess.Popen(analysis_cmd, shell=True)
(output, err) = p.communicate()


In [ ]:
### Visualizing and filtering AF2 results
AF2_DIR = f"{WDIR}/inpainting/5.1_2nd_af2"
os.chdir(AF2_DIR)

scores_af2 = pd.read_csv("scores.sc", sep="\s+", header=0)

### Filtering AF2 scores based on lddt and rmsd
# Define your desired cutoffs here:
AF2_filters = {"lDDT": [85.0, ">="],
               "rmsd": [1.2, "<="],
               "rmsd_SR1": [1.0, "<="]}  # 1st catalytic residue sc-rmsd

scores_af2_filtered = utils.filter_scores(scores_af2, AF2_filters)
utils.dump_scorefile(scores_af2_filtered, "filtered_scores.sc")

## Plotting AF2 scores
plt.figure(figsize=(12, 3))
for i,k in enumerate(AF2_filters):
    plt.subplot(1, 3, i+1)
    plt.hist(scores_af2[k])
    plt.title(k)
    plt.xlabel(k)
plt.tight_layout()
plt.show()
    
utils.plot_score_pairs(scores_af2, "lDDT", "rmsd", AF2_filters["lDDT"][0], AF2_filters["rmsd"][0])

In [ ]:
### Copying good predictions to a separate directory
os.chdir(AF2_DIR)

if len(scores_af2_filtered) > 0:
    os.makedirs("good", exist_ok=True)
    good_af2_models = [row["Output_PDB"]+".pdb" for idx,row in scores_af2_filtered.iterrows()]
    for pdb in good_af2_models:
        copy2(pdb, f"good/{pdb}")
    good_af2_models = glob.glob(f"{AF2_DIR}/good/*.pdb")
else:
    sys.exit("No good models to continue this pipeline with")

os.chdir(f"{AF2_DIR}/good")


## 6.1: Final FastRelax with the ligand
Relaxing good AF2 models together with the ligand

In [ ]:
AF2_DIR = f"{WDIR}/inpainting/5.1_2nd_af2"
DESIGN_DIR_ligMPNN = f"{WDIR}/inpainting/3.1_design_pocket_ligandMPNN"
assert len(glob.glob(f"{AF2_DIR}/good/*.pdb")) > 0, "No good AF2 models to relax with"

os.chdir(WDIR)
RELAX_DIR = f"{WDIR}/inpainting/6.1_final_relax"
os.makedirs(RELAX_DIR, exist_ok=True)
os.chdir(RELAX_DIR)


## First matching up the AF2 output filenames of step 5 with pocket design filenames from step 3
ref_and_model_pairs = []
for r in glob.glob(f"{DESIGN_DIR_ligMPNN}/good/*.pdb"):
    for pdbfile in glob.glob(f"{AF2_DIR}/good/*.pdb"):
        if os.path.basename(r).replace(".pdb", "_") in pdbfile:
            ref_and_model_pairs.append((r, pdbfile))

assert len(ref_and_model_pairs) == len(glob.glob(f"{AF2_DIR}/good/*.pdb")), "Was not able to match all models with reference structures"


## Generating commands for relax jobs
### Performing 1 relax iteration on each input structure
NSTRUCT = 1
cstfile = f"{SCRIPT_DIR}/theozyme/HBA/HBA_CYS_UPO.cst"

commands_relax = []
cmds_filename_rlx = "commands_design"
os.makedirs("logs",exist_ok=True)

with open(cmds_filename_rlx, "w") as file:
    for r_m in ref_and_model_pairs:
        commands_relax.append(f"{PYTHON['general']} {SCRIPT_DIR}/scripts/design/align_add_ligand_relax.py "
                              f"--outdir ./ --ligand {LIGAND} --ref_pdb {r_m[0]} "
                              f"--pdb {r_m[1]} --nstruct {NSTRUCT} "
                              f"--params {' '.join(params)} --cstfile {cstfile} > logs/{os.path.basename(pdb).replace('.pdb', '.log')}\n")
        file.write(commands_relax[-1])

print("Example design command:")
print(commands_relax[-1])
print(f"Total # of commands:{len(commands_relax)}")


### Running design jobs with Slurm.
submit_script = "submit_relax.sh"
utils.create_slurm_submit_script(filename=submit_script, name="6.1_final_relax", mem="4g", 
                                 N_cores=1, time="0:30:00", email=EMAIL, array=len(commands_relax),
                                 array_commandfile=cmds_filename_rlx)

print(f"current working directory:{os.getcwd()}\n")

if not os.path.exists(RELAX_DIR+"/.done"):
    for command_i in commands_relax:
        print(command_i)
        p = subprocess.Popen(command_i, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
        (output, err) = p.communicate()
        print(f"err:\n{err}\n")

### Analyzing final relaxed structures
Filtering them based on the same metrics as was used for the initial design

In [ ]:
## Analyzing Rosetta designs
RELAX_DIR = f"{WDIR}/inpainting/6.1_final_relax"
os.chdir(RELAX_DIR)

scores = pd.read_csv("scorefile.txt", sep="\s+", header=0)

filters = {'all_cst': [1.0, '<='],
 'nlr_SR1_rms': [0.8, '<='],
 'nlr_totrms': [1.0, '<='],
 'L_SASA': [0.2, '<='],
 'COO_hbond': [1.0, '='],
 'heme_angle_wrst': [80.0, '>='],
 'score_per_res': [0.0, '<='],
 'corrected_ddg': [-50.0, '<='],
 'cms_per_atom': [4.8, '>='],
 'rmsd_CA_rlx_in': [1.0, "<="]}  # rmsd_CA_rlx_in is rmsd between relaxed structure and AF2 prediction

filtered_scores = utils.filter_scores(scores, filters)

## Plotting relax scores
plt.figure(figsize=(12, 9))
for i,k in enumerate(filters):
    if k not in scores.keys():
        continue
    plt.subplot(4, 3, i+1)
    plt.hist(scores[k])
    plt.title(k)
    plt.xlabel(k)
plt.tight_layout()
plt.show()

### Copying good designs over to a new directory
if len(filtered_scores) > 0:
    os.makedirs(f"{RELAX_DIR}/good", exist_ok=True)
    for idx, row in filtered_scores.iterrows():
        copy2(row["description"]+".pdb", "good/"+row["description"]+".pdb")
else:
    print("No good designs created, bummer...")

In [ ]:
if len(filtered_scores) > 0:
    print(f"CONGRATULATIONS! You have successfully designed {len(filtered_scores)} proteins against ligand {LIGAND}")
    print("You can find the design models in the directory:\n"
          f"    {RELAX_DIR}/good")
    print("\nIt is advised you manually inspect them before ordering.")